In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [4]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(
    question,
    num_results=5
)

In [6]:
search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
from homework_rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [9]:
answer = assistant.rag('How does the agentic loop keep calling the model until it stops?')
print(answer)

The loop keeps calling the model by checking whether the model returned any `function_call` items.

- It sends the current `messages` history to the model.
- If the response includes a function call, the code runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`.
- Then it repeats the `while True` loop.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is simple: **no function calls this turn means the model is done and the loop ends**.


In [10]:
usage = assistant.get_usage()

In [11]:
usage

ResponseUsage(input_tokens=7095, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=122, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7217)

In [12]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

In [15]:
from minsearch import Index

chunk_index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
chunk_index.fit(chunks)

In [16]:
chunk_assistant = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
    instructions=instructions,
)

In [17]:
answer = chunk_assistant.rag('How does the agentic loop keep calling the model until it stops?')
print(answer)

The loop keeps calling the model inside a `while True` loop. Each turn:

1. It sends the current `messages` to the model.
2. It checks `response.output` for any `function_call` items.
3. If there is a function call, it runs the tool, appends the result to `messages`, and sets `has_function_calls = True`.
4. If there are no function calls in that turn, it breaks out of the loop.

So the exit condition is: **no function calls this turn means the loop stops**.


In [18]:
chunk_assistant.get_usage()

ResponseUsage(input_tokens=2278, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=118, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2396)

In [19]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [20]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5,
    )

In [21]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [22]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [23]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [25]:
intructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [27]:
result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback,
)

-> Response received


-> Response received
